# Explainable AI (XAI): TFT Variable Selection Weights
**Objective:** A predictive model is useless to a risk committee if it operates as a black box. The Temporal Fusion Transformer (TFT) utilizes a specialized architecture called a **Variable Selection Network (VSN)**. 

By extracting the attention weights from the VSN, we can mathematically prove exactly which macroeconomic indicators (Oil, S&P 500, Port Congestion, or News Sentiment) the neural network relied on to make its 7-day forecast.

In [ ]:
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt
from pytorch_forecasting import TemporalFusionTransformer
import sys

# Add the src directory to the python path so we can import our dataset builder
sys.path.append(os.path.abspath('../src'))
from models.dataset import BDIDatasetBuilder

# 1. Load the Production Model into RAM
model_path = "../models/best_tft_model.ckpt"
model = TemporalFusionTransformer.load_from_checkpoint(model_path)
model.eval()

# 2. Load the data exactly as the model expects it
df = pd.read_parquet('../data/processed/master_feature_set.parquet')
builder = BDIDatasetBuilder()
df_prepped = builder.prep_dataframe_for_pytorch(df)

# Create the validation dataset to pull a sample from
val_split = int(len(df_prepped) * 0.90)
train_ds, _ = builder.create_dataset()
val_ds = train_ds.from_dataset(train_ds, df_prepped[val_split:], stop_randomization=True)
val_dataloader = val_ds.to_dataloader(train=False, batch_size=32, num_workers=0)

# 3. Ask the AI to Interpret Itself
# PyTorch Forecasting has built-in XAI interpretation methods
raw_predictions = model.predict(val_dataloader, mode="raw", return_x=True)
interpretation = model.interpret_output(raw_predictions.output, reduction="sum")

# 4. Plot the Variable Importance
fig, ax = plt.subplots(figsize=(10, 6))

# We extract the weights for the unknown time-varying inputs (our macro indicators)
importance_dict = interpretation["encoder_variables"]
variables = list(importance_dict.keys())
weights = [importance_dict[var].item() for var in variables]

# Sort for a clean chart
sorted_indices = np.argsort(weights)
sorted_vars = [variables[i] for i in sorted_indices]
sorted_weights = [weights[i] for i in sorted_indices]

ax.barh(sorted_vars, sorted_weights, color='#1f77b4')
ax.set_title("AI Brain Scan: Macroeconomic Feature Importance", fontsize=14, fontweight='bold')
ax.set_xlabel("Relative Attention Weight (How much the AI cared)", fontsize=12)
ax.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()